# Getting Third-Party Data into ATLAS via the Open Streaming Architecture

This notebook helps you set up the **ATLAS Open Streaming Architecture** infrastructure locally and write your own data producer — the typical workflow when you need to get **third-party or custom data into ATLAS**.

## When would I use this?

There are two main streaming scenarios in ATLAS:

| Scenario | Producer | Consumer | Custom code needed? |
|----------|----------|----------|-------------------|
| **Live car data** | ADS -> Bridge Service (auto) | ATLAS Stream Recorder | No — just config |
| **Third-party data** | **Your code (this notebook)** | ATLAS Stream Recorder | Yes — you write the producer |

The **Bridge Service** handles live telemetry from an ADS automatically — you just enable Remote Data Feed in ADS and the Bridge Service decodes and publishes to Kafka for you.

But when you have **your own sensors, simulations, external data sources, or post-processing pipelines**, you need to write a producer that publishes data through the Stream API. That's what this notebook demonstrates.

## What this notebook covers
1. **Start the infrastructure** — Kafka, Stream API, Key Generator (one command)
2. **Write a producer** — publish parameter data into Kafka via the Stream API
3. **Verify the data** — quick plot of what was published
4. **Connect ATLAS** — point the Stream Recorder at your local Kafka to view in ATLAS

## Architecture
```
                                                          ┌──────────────────┐
┌─────────────────┐    gRPC     ┌──────────────┐  Kafka   │  ATLAS           │
│  Your Producer   │ ─────────>│  Stream API   │ ───────>│  Stream Recorder  │
│  (this notebook) │  :13579    │  Server       │  :9094   │  (view in ATLAS) │
└─────────────────┘             └──────────────┘          └──────────────────┘
                                       │
                                ┌──────────────┐
                                │ Key Generator │
                                │   :15379      │
                                └──────────────┘
```

## Prerequisites
- **Docker Desktop** running
- **Python 3.8+**
- **Git**

## Configuration

Adjust these before running. The defaults work out of the box for a local setup.

In [ ]:
# ============================================================
# CONFIGURATION - Change these as needed
# ============================================================

STREAM_API_ADDRESS = "localhost:13579"       # Stream API gRPC endpoint
KEY_GENERATOR_ADDRESS = "localhost:15379"     # Key Generator gRPC endpoint
KAFKA_BROKER = "localhost:9094"              # Kafka broker (for health check only)

DATA_SOURCE = "Default"                      # Data source name
STREAM = "Stream1"                           # Stream to write data to
SAMPLE_FREQUENCY_HZ = 100                    # Frequency of generated samples (Hz)
NUM_SECONDS_TO_STREAM = 10                   # How many seconds of data to generate

## Part 1: Start the Streaming Infrastructure

This starts the full stack with one command: Kafka, Stream API, Key Generator, and a Kafka UI for debugging.

> **Note:** This same infrastructure is what the Bridge Service connects to when streaming live car data. If you're setting up for ADS + Bridge Service, you still need this running — the only difference is the Bridge Service acts as the producer instead of your code.

After running, you can inspect Kafka topics at **http://localhost:8080**

In [ ]:
import subprocess, time, os

os.chdir(os.path.dirname(os.path.abspath("__file__")) if os.path.exists("docker-compose.yml") else ".")

print("Starting Docker Compose stack...")
result = subprocess.run(
    ["docker", "compose", "up", "-d", "--wait"],
    capture_output=True, text=True, timeout=120
)
print(result.stdout)
if result.returncode != 0:
    print(f"STDERR: {result.stderr}")
    raise RuntimeError("Docker Compose failed to start. Is Docker Desktop running?")

print("Waiting for services to be ready...")
time.sleep(5)

# Verify services
result = subprocess.run(["docker", "compose", "ps"], capture_output=True, text=True)
print(result.stdout)

## Part 2: Install Dependencies & Generate gRPC Client

The Stream API uses gRPC. This step clones the proto definitions from GitHub and generates Python client stubs so you can talk to the Stream API from this notebook.

In [ ]:
import subprocess, sys, os

# Install Python dependencies
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "grpcio", "grpcio-tools", "numpy", "pandas", "matplotlib"])

# Clone the proto definitions if not already present
PROTO_REPO = "https://github.com/Software-Products/MA.DataPlatforms.Protocol.git"
PROTO_DIR = "MA.DataPlatforms.Protocol"

if not os.path.exists(PROTO_DIR):
    print("Cloning proto definitions...")
    subprocess.check_call(["git", "clone", "--depth", "1", PROTO_REPO])
else:
    print("Proto repo already cloned.")

# Find the proto root (contains the 'ma' directory structure)
proto_root = os.path.join(PROTO_DIR, "proto")
if not os.path.exists(proto_root):
    proto_root = PROTO_DIR  # fallback

print(f"Proto root: {proto_root}")
print("Contents:", os.listdir(proto_root))

In [ ]:
import glob as globmod
import subprocess

# Find all .proto files
proto_files = globmod.glob(os.path.join(proto_root, "**", "*.proto"), recursive=True)
print(f"Found {len(proto_files)} proto files:")
for f in proto_files:
    print(f"  {f}")

# Sort so that dependencies (open_data) compile before dependents (api)
proto_files.sort(key=lambda x: (0 if "open_data" in x else 1 if "key_generator" in x else 2))

# Generate Python gRPC stubs
output_dir = os.getcwd()
for proto_file in proto_files:
    rel_path = os.path.relpath(proto_file, proto_root)
    print(f"\nGenerating stubs for: {rel_path}")
    result = subprocess.run([
        sys.executable, "-m", "grpc_tools.protoc",
        f"-I{proto_root}",
        f"--python_out={output_dir}",
        f"--pyi_out={output_dir}",
        f"--grpc_python_out={output_dir}",
        proto_file,
    ], capture_output=True, text=True)
    if result.returncode != 0:
        print(f"  WARNING: {result.stderr}")
    else:
        print(f"  OK")

# Create __init__.py files so Python treats generated dirs as packages
ma_root = os.path.join(output_dir, "ma")
for dirpath, dirnames, filenames in os.walk(ma_root):
    init_file = os.path.join(dirpath, "__init__.py")
    if not os.path.exists(init_file):
        open(init_file, "w").close()
        print(f"  Created {os.path.relpath(init_file)}")

# Add current directory to Python path so imports work
if output_dir not in sys.path:
    sys.path.insert(0, output_dir)

print("\nProto stubs generated successfully!")

## Part 3: Connect & Verify

Quick check that the Stream API and Key Generator are reachable.

In [ ]:
import grpc
from ma.streaming.api.v1 import api_pb2, api_pb2_grpc
from ma.streaming.key_generator.v1 import key_generator_pb2, key_generator_pb2_grpc
from ma.streaming.open_data.v1 import open_data_pb2

# Connect to Stream API
stream_channel = grpc.insecure_channel(STREAM_API_ADDRESS)
session_stub = api_pb2_grpc.SessionManagementServiceStub(stream_channel)
connection_stub = api_pb2_grpc.ConnectionManagerServiceStub(stream_channel)
data_format_stub = api_pb2_grpc.DataFormatManagerServiceStub(stream_channel)
packet_writer_stub = api_pb2_grpc.PacketWriterServiceStub(stream_channel)
packet_reader_stub = api_pb2_grpc.PacketReaderServiceStub(stream_channel)

# Connect to Key Generator
keygen_channel = grpc.insecure_channel(KEY_GENERATOR_ADDRESS)
keygen_stub = key_generator_pb2_grpc.UniqueKeyGeneratorServiceStub(keygen_channel)

# Test connectivity - generate a key
try:
    key_response = keygen_stub.GenerateUniqueKey(
        key_generator_pb2.GenerateUniqueKeyRequest(
            type=key_generator_pb2.KEY_TYPE_STRING
        )
    )
    print(f"Key Generator connected - test key: {key_response.string_key}")
except grpc.RpcError as e:
    print(f"Key Generator connection failed: {e.details()}")
    print("Make sure Docker Compose is running (Step 1)")

print("Stream API channel ready on", STREAM_API_ADDRESS)

## Part 4: Write Your Own Producer

This is the core of the notebook. Here we act as a **third-party data producer** — creating a session, defining parameters, and streaming data into Kafka via the Stream API.

The example publishes sine and cosine waves, but you'd replace this with your own data source (sensors, simulation output, external feeds, etc.).

### The producer lifecycle:
1. **Create a session** — tells the Stream API a new data stream is starting
2. **Publish configuration** — define your parameters (names, units, frequencies, etc.)
3. **Register data format** — maps parameter names to a format ID
4. **Stream data packets** — publish your actual data in chunks
5. **End session** — signals completion

In [ ]:
import datetime, time
import numpy as np
from google.protobuf import wrappers_pb2, any_pb2

# --- Helper functions ---

def get_packet_type(packet_content) -> str:
    """Get the type string for an open_data Packet."""
    packet_type = packet_content.__class__.__name__
    if packet_type.endswith("Packet"):
        packet_type = packet_type[:-6]
    return packet_type

def write_packet(stub, packet_content, packet_type, data_source, session_key, stream, is_essential=False):
    """Write a single packet to a stream via the Stream API."""
    message = open_data_pb2.Packet(
        type=packet_type,
        content=packet_content,
        session_key=session_key,
        is_essential=is_essential,
    )
    stub.WriteDataPacket(
        request=api_pb2.WriteDataPacketRequest(
            detail=api_pb2.DataPacketDetails(
                message=message,
                data_source=data_source,
                stream=stream,
                session_key=session_key,
            )
        )
    )

def build_double_column(samples, status=open_data_pb2.DataStatus.DATA_STATUS_VALID):
    """Build a SampleColumn from a list of float values."""
    constructed = [open_data_pb2.DoubleSample(value=s, status=status) for s in samples]
    return open_data_pb2.SampleColumn(
        double_samples=open_data_pb2.DoubleSampleList(samples=constructed)
    )

print("Helper functions defined.")

In [ ]:
# ============================================================
# CREATE SESSION & STREAM DATA
# ============================================================

PARAM_IDENTIFIERS = ["Sin:MyApp", "Cos:MyApp"]
MAIN_STREAM = ""
interval_ns = int(1e9 / SAMPLE_FREQUENCY_HZ)

# 1. Create a new session
print("Creating session...")
create_response = session_stub.CreateSession(
    api_pb2.CreateSessionRequest(
        data_source=DATA_SOURCE,
        identifier=f"Notebook Session {datetime.datetime.now():%Y-%m-%d %H:%M:%S}",
        type="Session",
        version=1,
    )
)
session_key = create_response.session_key
print(f"Session created: {session_key}")

# 2. Publish NewSession packets to the streams
new_session_pkt = open_data_pb2.NewSessionPacket(data_source=DATA_SOURCE)
pkt_type = get_packet_type(new_session_pkt)
write_packet(packet_writer_stub, new_session_pkt.SerializeToString(), pkt_type, DATA_SOURCE, session_key, STREAM)
write_packet(packet_writer_stub, new_session_pkt.SerializeToString(), pkt_type, DATA_SOURCE, session_key, MAIN_STREAM)
print("NewSession packets published.")
time.sleep(2)

# 3. Publish configuration packet
config_id = keygen_stub.GenerateUniqueKey(
    key_generator_pb2.GenerateUniqueKeyRequest(type=key_generator_pb2.KEY_TYPE_STRING)
).string_key

param_defs = []
for pid in PARAM_IDENTIFIERS:
    name, app = pid.split(":")
    param_defs.append(open_data_pb2.ParameterDefinition(
        identifier=pid, name=name, application_name=app,
        description="", units="m",
        data_type=open_data_pb2.DATA_TYPE_FLOAT64,
        format_string="%6.3f",
        frequencies=[SAMPLE_FREQUENCY_HZ],
        max_value=1, min_value=-1,
    ))

config_pkt = open_data_pb2.ConfigurationPacket(
    config_id=config_id,
    parameter_definitions=param_defs,
    group_definitions=[open_data_pb2.GroupDefinition(
        identifier="MyApp", application_name="MyApp",
        name="MyApp", description="MyApp", groups=[],
    )],
)
pkt_type = get_packet_type(config_pkt)
write_packet(packet_writer_stub, config_pkt.SerializeToString(), pkt_type, DATA_SOURCE, session_key, MAIN_STREAM, is_essential=True)
print(f"Configuration published (config_id: {config_id})")

# 4. Register data format
data_format_response = data_format_stub.GetParameterDataFormatId(
    request=api_pb2.GetParameterDataFormatIdRequest(
        data_source=DATA_SOURCE, parameters=PARAM_IDENTIFIERS,
    )
)
data_format_id = data_format_response.data_format_identifier
print(f"Data format registered (id: {data_format_id})")

# 5. Stream data!
print(f"\nStreaming {NUM_SECONDS_TO_STREAM}s of sine/cosine data at {SAMPLE_FREQUENCY_HZ} Hz...")
current_time_ns = time.time_ns()
all_timestamps = []
all_sin = []
all_cos = []

for sec in range(NUM_SECONDS_TO_STREAM):
    t = current_time_ns
    time_points = np.arange(0, 1e9, interval_ns)
    sin_vals = [float(np.sin((t + i) / 1e9)) for i in time_points]
    cos_vals = [float(np.cos((t + i) / 1e9)) for i in time_points]

    # Store for plotting later
    all_timestamps.extend([t + i for i in time_points])
    all_sin.extend(sin_vals)
    all_cos.extend(cos_vals)

    data_pkt = open_data_pb2.PeriodicDataPacket(
        data_format=open_data_pb2.SampleDataFormat(data_format_identifier=data_format_id),
        start_time=t,
        interval=interval_ns,
        columns=[build_double_column(sin_vals), build_double_column(cos_vals)],
    )
    pkt_type = get_packet_type(data_pkt)
    write_packet(packet_writer_stub, data_pkt.SerializeToString(), pkt_type, DATA_SOURCE, session_key, STREAM)

    current_time_ns = t + int(1e9)
    print(f"  Chunk {sec+1}/{NUM_SECONDS_TO_STREAM} published ({len(sin_vals)} samples)")
    time.sleep(0.5)  # Small delay between chunks

print(f"\nDone! Published {len(all_sin)} total samples.")

## Part 5: Verify — Plot the Streamed Data

Quick sanity check that the data we published looks right.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Convert to relative time in seconds for cleaner plotting
t0 = all_timestamps[0]
time_sec = [(t - t0) / 1e9 for t in all_timestamps]

df = pd.DataFrame({"time_s": time_sec, "Sin": all_sin, "Cos": all_cos})

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(df["time_s"], df["Sin"], linewidth=0.5, color="tab:blue")
ax1.set_ylabel("Sin:MyApp")
ax1.set_title("Streamed Data via ATLAS Stream API + Kafka")
ax1.grid(True, alpha=0.3)

ax2.plot(df["time_s"], df["Cos"], linewidth=0.5, color="tab:orange")
ax2.set_ylabel("Cos:MyApp")
ax2.set_xlabel("Time (seconds)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"DataFrame shape: {df.shape}")
df.head(10)

## Part 6: View in ATLAS with the Stream Recorder

Now that your data is in Kafka, you can consume it in ATLAS using the **Stream Recorder**. No custom consumer code needed — ATLAS handles this natively.

### Setup steps:

1. **Open ATLAS** on the same machine (or one that can reach `localhost:9094`)

2. **Open the Stream Recorder**
   - Go to **Recording** > **Stream Recorder** (or find it in the toolbar)

3. **Configure the connection**
   - Set the **Kafka broker** to: `localhost:9094`
   - The Stream Recorder will auto-discover available sessions and streams

4. **Start recording**
   - You should see the session created by this notebook appear
   - Select it and the Stream Recorder will begin consuming packets from Kafka
   - Your parameters (`Sin:MyApp`, `Cos:MyApp`) will appear in the parameter tree

5. **View live data**
   - Add parameters to a display page to see them plot in real time
   - If the session has already ended, the data will replay from the beginning

> **Tip:** If you're running this notebook and ATLAS at the same time, run the producer cell (Part 4) again *without* ending the session — the Stream Recorder will display the data as it arrives in real time.

### For ADS + Bridge Service users

If you're streaming from an ADS instead of custom code, the setup is similar:
1. Enable **Remote Data Feed** in ADS: Tools > Options > Recording > General > set to "True"
2. Restart ADS — the Bridge Service launches automatically
3. Configure the Bridge Service's `BridgeServiceConfig.json` to point at your Kafka broker (`localhost:9094`)
4. Open ATLAS Stream Recorder and connect to the same Kafka broker — live car data will appear automatically

## Part 7: End Session & Cleanup

Run this when you're done. If you want ATLAS to keep seeing the session, skip the "End session" cell and just close the notebook — the data stays in Kafka until you tear down Docker.

In [ ]:
# End the session gracefully
# end_pkt = open_data_pb2.EndOfSessionPacket(data_source=DATA_SOURCE)
# pkt_type = get_packet_type(end_pkt)
# write_packet(packet_writer_stub, end_pkt.SerializeToString(), pkt_type, DATA_SOURCE, session_key, STREAM)
# write_packet(packet_writer_stub, end_pkt.SerializeToString(), pkt_type, DATA_SOURCE, session_key, MAIN_STREAM)

# session_stub.EndSession(
#     api_pb2.EndSessionRequest(data_source=DATA_SOURCE, session_key=session_key)
# )
# print(f"Session {session_key} ended.")

# # Close gRPC channels
# stream_channel.close()
# keygen_channel.close()
print("gRPC channels closed.")

In [ ]:
# Tear down Docker Compose stack (stops Kafka, Stream API, Key Generator, Kafka UI)
# Only run this when you're completely done — ATLAS won't be able to connect after this.

# !docker compose down -v

print("To stop the infrastructure, run:  docker compose down -v")
print("To restart it later, run:         docker compose up -d --wait")
print("")
print("Kafka UI:       http://localhost:8080")
print("Stream API:     localhost:13579 (gRPC)")
print("Kafka broker:   localhost:9094")